# Stage1 分钟接收器失败只读归因

复用两个封存 Qlib Recorder 与八条旧路线结论。核对标签日内可辨性、分钟字段变化和预测到动作传递；零新拟合、零新账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects

root = Path.cwd()
card = root / "research/stage1_minute_receiver_failure_audit_20261028"
plan_bytes = (card / "plan.json").read_bytes()
plan = json.loads(plan_bytes)
label_card = root / plan["source_cards"]["labels"]
info_card = root / plan["source_cards"]["information"]
label_summary = json.loads((label_card / "summary.json").read_text())
info_summary = json.loads((info_card / "summary.json").read_text())
qlib.init(provider_uri=info_summary["provider_uri"], region="us", kernels=1)
labels_obj, label_receipt = load_verified_recorder_objects(
    label_card, expected_recorder_id=label_summary["recorder_id"], artifacts=["analysis/labels"])
info_obj, info_receipt = load_verified_recorder_objects(
    info_card, expected_recorder_id=info_summary["recorder_id"],
    artifacts=["analysis/predictions", "analysis/information_diagnostic"])
frame = labels_obj["analysis/labels"]
frame = frame.loc[frame.funded_change].sort_values(["trade_date", "decision_at"]).copy()
assert len(frame) == 271060 and frame.trade_date.nunique() == 736
by_year = []
for year in plan["years"]:
    item = frame.loc[frame.trade_date.dt.year.eq(year)]
    if item.empty:
        continue
    groups = item.groupby("trade_date")
    y = item.local_advantage
    within = y - groups.local_advantage.transform("mean")
    fields = {}
    for field in plan["minute_fields"]:
        x = item[field] - groups[field].transform("mean")
        fields[field] = {
            "days_with_intraday_variation": int(groups[field].max().gt(groups[field].min()).sum()),
            "within_day_pearson_descriptive": float(x.corr(within))
        }
    by_year.append({"year": year, "days": int(item.trade_date.nunique()),
        "funded_rows": len(item), "within_day_label_variance_share": float(within.var() / y.var()),
        "days_with_label_intraday_variation": int(groups.local_advantage.max().gt(groups.local_advantage.min()).sum()),
        "minute_fields": fields})
score_audit = []
for year in [2021, 2022]:
    candidate = info_obj["analysis/predictions"][f"{year}_candidate"]
    control = info_obj["analysis/predictions"][f"{year}_control"]
    assert candidate.trade_date.equals(control.trade_date)
    assert candidate.decision_at.equals(control.decision_at)
    difference = (candidate.score - control.score).abs()
    group = candidate.groupby("trade_date")
    score_range = group.score.max() - group.score.min()
    first = candidate.loc[candidate.score.gt(0)].sort_values(["trade_date", "decision_at"]).groupby("trade_date").head(1)
    first_eligible = group.decision_at.min().reindex(first.trade_date).to_numpy()
    score_audit.append({"year": year, "evaluation_days": int(candidate.trade_date.nunique()),
        "different_candidate_control_scores": int(difference.gt(0).sum()),
        "max_candidate_control_score_gap": float(difference.max()),
        "days_with_constant_candidate_score": int(score_range.eq(0).sum()),
        "trigger_days": int(first.trade_date.nunique()),
        "triggers_at_first_eligible_minute": int(first.decision_at.eq(first_eligible).sum())})
source_review = [{"path": path, "sha256": hashlib.sha256((root/path).read_bytes()).hexdigest()}
                 for path in plan["source_review"]]
result = {"experiment": plan["experiment"], "plan_sha256": hashlib.sha256(plan_bytes).hexdigest(),
    "source_recorder_ids": {"labels": label_summary["recorder_id"], "information": info_summary["recorder_id"]},
    "source_receipt_entries": {"labels": len(label_receipt), "information": len(info_receipt)},
    "funded_rows": len(frame), "funded_days": int(frame.trade_date.nunique()),
    "by_year": by_year, "score_audit": score_audit,
    "saved_split_diagnostic": info_obj["analysis/information_diagnostic"],
    "reviewed_closed_conclusion_hashes": source_review,
    "new_model_fits": 0, "new_native_accounts": 0, "new_recorders": 0,
    "historical_research_passed": False,
    "decision": "read_only_attribution_no_minute_action_information_proven"}
(card / "audit.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
print(json.dumps({"funded_days": result["funded_days"], "by_year": [
    {"year": x["year"], "within_day_label_variance_share": x["within_day_label_variance_share"]}
    for x in by_year], "score_audit": score_audit}, ensure_ascii=False, indent=2))

[34018:MainThread](2026-10-28 17:50:25,074) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[34018:MainThread](2026-10-28 17:50:25,436) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[34018:MainThread](2026-10-28 17:50:25,436) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "funded_days": 736,
  "by_year": [
    {
      "year": 2019,
      "within_day_label_variance_share": 0.07101512189018205
    },
    {
      "year": 2020,
      "within_day_label_variance_share": 0.09656754517301631
    },
    {
      "year": 2021,
      "within_day_label_variance_share": 0.0936847125992612
    },
    {
      "year": 2022,
      "within_day_label_variance_share": 0.187195467566507
    }
  ],
  "score_audit": [
    {
      "year": 2021,
      "evaluation_days": 219,
      "different_candidate_control_scores": 0,
      "max_candidate_control_score_gap": 0.0,
      "days_with_constant_candidate_score": 198,
      "trigger_days": 31,
      "triggers_at_first_eligible_minute": 31
    },
    {
      "year": 2022,
      "evaluation_days": 41,
      "different_candidate_control_scores": 0,
      "max_candidate_control_score_gap": 0.0,
      "days_with_constant_candidate_score": 18,
      "trigger_days": 5,
      "triggers_at_first_eligible_minute": 4
    }
  ]
}


## 中文结论

复用封存 Qlib Recorder 的 271060 条整股标签、736 个可行动日；没有新拟合或账户。2019–2022 局部标签日内方差仅占总方差的 7.10%–18.72%。2021/2022 候选与控制的 96020 条预测完全一致，所有分钟输入分裂为零；31/31 与 4/5 次触发在当天第一条可行动分钟。固定事件模型主要成为账户状态的开盘附近判断，未证明分钟事件信息。

逐项核对八条旧路线的失败和适用边界，详见 `conclusion.md` 与 `audit.json`。决定 `read_only_attribution_no_minute_action_information_proven`，SOXX 历史 Goal 未通过。下一步仅在逐年点时 f696+b472 上先建立每天一次 09:46 合法扣费配置标签，再冻结新信息门；历史分钟首次送达仍是 `needs_data`。